# Eksperimen 21C (EXP21C): Multi-Window Temporal Splitting & Walk-Forward Validation
### Google Colab Execution Notebook

Notebook ini memuat seluruh pipeline `EXP21C`:
1. **Multi-Horizon Rolling Walk-Forward Validation (SPLIT-2)**: Window 1 (2017–2018), Window 2 (2018–2019), Window 3 (2019–2020).
2. **Strict Out-of-Sample Preprocessing (SPLIT-1)**: Scaler dan kamus kategori fit strictly di Train window masing-masing.
3. **Embargo Gap Ablation (SPLIT-3)**: Pengujian variasi 0 hari, 8 hari, dan 30 hari jeda investigasi fraud.
4. **GraphSAGE Lift Benchmark**: Tabular Only vs GraphSAGE Only vs Hybrid lintas era.
5. **Visualisasi Publikasi 300 DPI**: Multi-window comparison, kurva PR/ROC, dan embargo sensitivity.

In [ ]:
# Mount Google Drive untuk mengakses dataset
from google.colab import drive
drive.mount('/content/drive')

# Cek status GPU
!nvidia-smi

In [ ]:
# Install PyTorch Geometric dan dependensi pendukung
!pip install torch-geometric xgboost pyyaml matplotlib pandas numpy scikit-learn


In [ ]:
%%writefile config.colab.yaml
data:
  transactions: "/content/drive/MyDrive/dataset/credit_card_transactions-ibm_v2.csv"
  max_rows: null
  start_year: 2016

time_windows:
  windows:
    - id: "window_1"
      name: "Window 1 (2017–2018)"
      reference_date: "2017-01-01"
      train_days: 300
      val_days: 30
      gap_days: 8
      test_days: 60
    - id: "window_2"
      name: "Window 2 (2018–2019)"
      reference_date: "2018-01-01"
      train_days: 300
      val_days: 30
      gap_days: 8
      test_days: 60
    - id: "window_3"
      name: "Window 3 (2019–2020)"
      reference_date: "2019-01-01"
      train_days: 300
      val_days: 30
      gap_days: 8
      test_days: 60

  embargo_ablation:
    enabled: true
    target_window_id: "window_2"
    gap_durations: [0, 8, 30]

features:
  mode: "clean"
  use_cyclical_time: true
  exclude_high_cardinality: true

sampling:
  method: "uniform"
  undersampling_ratio: 1.0

model:
  hidden_channels: 128
  out_channels: 1
  num_layers: 2
  aggr: "mean"
  dropout: 0.2

training:
  batch_size: 1024
  epochs: 20
  learning_rate: 0.001
  weight_decay: 0.0001
  label_smoothing: 0.0
  grad_clip_norm: 5.0
  warmup_epochs: 0
  use_lr_scheduler: true
  lr_scheduler_factor: 0.5
  lr_scheduler_patience: 3
  early_stopping_patience: 7
  loss_weight_strategy: "sqrt"
  max_pos_weight: 50.0
  log_step_interval: 1
  num_neighbors: [15, 10]

classifier:
  type: "xgboost"
  n_estimators: 100
  max_depth: 4
  learning_rate: 0.05
  min_child_weight: 10
  subsample: 0.8
  colsample_bytree: 0.8
  scale_pos_weight_strategy: "sqrt"

threshold_calibration:
  selection_metric: "val_f1"
  min_precision_constraint: 0.10

ablation:
  feature_subsets: true

output:
  model_dir: "./model/exp21c"
  result_dir: "./result/exp21c"



In [ ]:
%%writefile config.py
import yaml
from pathlib import Path

def load_config(config_path: str = "config.reference.yaml") -> dict:
    """
    Memuat file konfigurasi YAML dan mengembalikannya sebagai dictionary Python.
    """
    path = Path(config_path)
    if not path.exists():
        raise FileNotFoundError(f"File konfigurasi tidak ditemukan di: {config_path}")
        
    with open(path, "r", encoding="utf-8") as f:
        config = yaml.safe_load(f)
        
    return config



In [ ]:
%%writefile data_preprocessing.py
import pandas as pd
import numpy as np
from datetime import timedelta
from sklearn.preprocessing import StandardScaler
from pathlib import Path

def load_raw_data_era_modern(
    file_path: str, 
    start_year: int = 2016, 
    nrows: int = None
) -> pd.DataFrame:
    """
    Memuat dataset mentah IBM Credit Card Transactions, memfilter transaksi era modern (Year >= start_year),
    melakukan parsing datetime kronologis, pembersihan format Amount, dan normalisasi label target.
    Fungsi ini dijalankan SATU KALI untuk seluruh rangkaian rolling windows pada EXP21C.
    """
    print(f"\n[DATA LOADER] Memuat dataset dari {file_path} (Filter: Year >= {start_year}, nrows={nrows})...")
    
    if start_year is not None:
        chunks = []
        collected_rows = 0
        chunksize = 1_000_000
        
        for chunk in pd.read_csv(file_path, chunksize=chunksize):
            filtered_chunk = chunk[chunk['Year'] >= start_year]
            if len(filtered_chunk) > 0:
                if nrows is not None:
                    remaining = nrows - collected_rows
                    if remaining <= 0:
                        break
                    filtered_chunk = filtered_chunk.iloc[:remaining]
                chunks.append(filtered_chunk)
                collected_rows += len(filtered_chunk)
                if nrows is not None and collected_rows >= nrows:
                    break
                    
        if not chunks:
            raise ValueError(f"Tidak ada baris yang ditemukan dengan Year >= {start_year} pada file {file_path}")
            
        df = pd.concat(chunks, ignore_index=True)
    else:
        df = pd.read_csv(file_path, nrows=nrows)

    print(f"[DATA LOADER] Memproses {len(df):,} baris transaksi era modern...")
    print("[DATA LOADER] Melakukan parsing datetime kronologis...")
    df['Time'] = df['Time'].fillna('00:00')
    df['datetime'] = pd.to_datetime(
        df['Year'].astype(str) + '-' + 
        df['Month'].astype(str).str.zfill(2) + '-' + 
        df['Day'].astype(str).str.zfill(2) + ' ' + 
        df['Time']
    )
    
    print("[DATA LOADER] Mengurutkan transaksi secara strictly chronologic...")
    df = df.sort_values('datetime').reset_index(drop=True)
    
    print("[DATA LOADER] Membersihkan Amount dan label 'Is Fraud?'...")
    df['Amount'] = df['Amount'].astype(str).str.replace('$', '', regex=False).str.replace(',', '', regex=False).astype(float)
    df['Is Fraud?'] = df['Is Fraud?'].map({'Yes': 1, 'No': 0})
    
    print(f"[DATA LOADER] Pemuatan selesai: Rentang {df['datetime'].min()} s/d {df['datetime'].max()} ({len(df):,} baris).\n")
    return df

def extract_window_splits(
    df: pd.DataFrame, 
    reference_date: str, 
    train_days: int = 300, 
    val_days: int = 30, 
    gap_days: int = 8, 
    test_days: int = 60,
    window_name: str = "Window"
) -> tuple:
    """
    Memotong dataset kronologis ke dalam 4 partisi temporal SHINE FRAUD II:
    - Train Window: T0 s/d T0 + train_days
    - Validation Window: T0 + train_days s/d + val_days
    - Embargo Gap: T_val_end s/d + gap_days (tidak dipakai training/evaluasi)
    - Test Window: T_gap_end s/d + test_days
    
    Menghitung statistik lengkap setiap partisi (transaksi, fraud, user, merchant).
    """
    t0 = pd.to_datetime(reference_date)
    t_val_start = t0 + timedelta(days=train_days)
    t_gap_start = t_val_start + timedelta(days=val_days)
    t_test_start = t_gap_start + timedelta(days=gap_days)
    t_test_end = t_test_start + timedelta(days=test_days)
    
    train_df = df[(df['datetime'] >= t0) & (df['datetime'] < t_val_start)].copy().reset_index(drop=True)
    val_df = df[(df['datetime'] >= t_val_start) & (df['datetime'] < t_gap_start)].copy().reset_index(drop=True)
    gap_df = df[(df['datetime'] >= t_gap_start) & (df['datetime'] < t_test_start)].copy().reset_index(drop=True)
    test_df = df[(df['datetime'] >= t_test_start) & (df['datetime'] < t_test_end)].copy().reset_index(drop=True)
    
    def compute_split_stats(name: str, split_df: pd.DataFrame, start_dt, end_dt):
        n_tx = len(split_df)
        if n_tx == 0:
            return {
                "name": name,
                "start_date": str(start_dt),
                "end_date": str(end_dt),
                "num_transactions": 0,
                "num_fraud": 0,
                "fraud_rate_pct": 0.0,
                "unique_users": 0,
                "unique_merchants": 0,
                "total_nodes": 0
            }
        n_fraud = int(split_df['Is Fraud?'].sum())
        n_users = int(split_df['User'].nunique())
        n_merchants = int(split_df['Merchant Name'].nunique())
        return {
            "name": name,
            "start_date": str(start_dt),
            "end_date": str(end_dt),
            "num_transactions": n_tx,
            "num_fraud": n_fraud,
            "fraud_rate_pct": round((n_fraud / n_tx) * 100, 4) if n_tx > 0 else 0.0,
            "unique_users": n_users,
            "unique_merchants": n_merchants,
            "total_nodes": n_tx + n_users + n_merchants
        }

    total_window_df = pd.concat([train_df, val_df, gap_df, test_df], ignore_index=True)
    
    stats_dict = {
        "window_name": window_name,
        "reference_date_T0": str(t0),
        "durations_days": {
            "train": train_days,
            "validation": val_days,
            "embargo_gap": gap_days,
            "test": test_days,
            "total_horizon": train_days + val_days + gap_days + test_days
        },
        "splits": {
            "train": compute_split_stats("Train", train_df, t0, t_val_start),
            "validation": compute_split_stats("Validation", val_df, t_val_start, t_gap_start),
            "embargo_gap": compute_split_stats("Embargo Gap", gap_df, t_gap_start, t_test_start),
            "test": compute_split_stats("Test", test_df, t_test_start, t_test_end),
            "total_window": compute_split_stats("Total Window", total_window_df, t0, t_test_end)
        }
    }
    
    print(f"[{window_name}] Partisi terbentuk:")
    print(f"  • Train: {len(train_df):,} tx ({stats_dict['splits']['train']['num_fraud']} fraud, {stats_dict['splits']['train']['fraud_rate_pct']:.3f}%) | {t0.strftime('%Y-%m-%d')} s/d {t_val_start.strftime('%Y-%m-%d')}")
    print(f"  • Val  : {len(val_df):,} tx ({stats_dict['splits']['validation']['num_fraud']} fraud, {stats_dict['splits']['validation']['fraud_rate_pct']:.3f}%) | {t_val_start.strftime('%Y-%m-%d')} s/d {t_gap_start.strftime('%Y-%m-%d')}")
    print(f"  • Gap  : {len(gap_df):,} tx ({stats_dict['splits']['embargo_gap']['num_fraud']} fraud) | {t_gap_start.strftime('%Y-%m-%d')} s/d {t_test_start.strftime('%Y-%m-%d')} ({gap_days} hari)")
    print(f"  • Test : {len(test_df):,} tx ({stats_dict['splits']['test']['num_fraud']} fraud, {stats_dict['splits']['test']['fraud_rate_pct']:.3f}%) | {t_test_start.strftime('%Y-%m-%d')} s/d {t_test_end.strftime('%Y-%m-%d')}")
    
    return train_df, val_df, test_df, gap_df, stats_dict

def apply_strict_anti_leakage_preprocessing(
    train_df: pd.DataFrame,
    val_df: pd.DataFrame,
    test_df: pd.DataFrame,
    gap_df: pd.DataFrame = None,
    feature_mode: str = "clean",
    use_cyclical_time: bool = True
) -> tuple:
    """
    Menerapkan Strict Anti-Leakage Preprocessing Protocol (SPLIT-1):
    1. StandardScaler Amount HANYA di-fit pada train_df, lalu mentransformasikan val_df dan test_df.
    2. Encoding kategorikal HANYA mempelajari kamus nilai unik dari train_df. Nilai baru pada val/test dipetakan ke 0 (<oov>).
    3. Mengeliminasi Zip dan Merchant City (mode: clean) untuk mencegah pohon keputusan menghafal ID kuasi.
    4. Menambahkan fitur waktu siklik (sin/cos jam & hari) yang tidak rentan memorization.
    """
    if feature_mode == "clean":
        cat_columns = ['Use Chip', 'Merchant State', 'MCC', 'Errors?']
    elif feature_mode == "clean_basic":
        cat_columns = ['Use Chip', 'MCC', 'Errors?']
    elif feature_mode == "legacy_leaky":
        cat_columns = ['Use Chip', 'Merchant City', 'Merchant State', 'Zip', 'MCC', 'Errors?']
    else:
        raise ValueError(f"Feature mode '{feature_mode}' tidak dikenal. Pilih 'clean', 'clean_basic', atau 'legacy_leaky'.")

    # A. Standarisasi Fitur Numerik (Amount) — FIT HANYA DI TRAIN WINDOW INI
    scaler_amount = StandardScaler()
    train_df['Amount'] = scaler_amount.fit_transform(train_df[['Amount']])
    val_df['Amount'] = scaler_amount.transform(val_df[['Amount']])
    test_df['Amount'] = scaler_amount.transform(test_df[['Amount']])
    if gap_df is not None and len(gap_df) > 0:
        gap_df['Amount'] = scaler_amount.transform(gap_df[['Amount']])

    # B. Robust Categorical Encoding — DICTIONARY FIT HANYA DI TRAIN WINDOW INI
    categorical_encoders = {}
    for col in cat_columns:
        train_df[col] = train_df[col].fillna('<missing>').astype(str)
        val_df[col] = val_df[col].fillna('<missing>').astype(str)
        test_df[col] = test_df[col].fillna('<missing>').astype(str)
        if gap_df is not None and len(gap_df) > 0:
            gap_df[col] = gap_df[col].fillna('<missing>').astype(str)
            
        unique_train_vals = sorted(train_df[col].unique().tolist())
        # Index 0 dicadangkan untuk OOV (<unknown>), kategori train mulai index 1
        cat_to_id = {val: idx + 1 for idx, val in enumerate(unique_train_vals)}
        categorical_encoders[col] = cat_to_id
        
        train_df[col] = train_df[col].map(cat_to_id).fillna(0).astype(int)
        val_df[col] = val_df[col].map(cat_to_id).fillna(0).astype(int)
        test_df[col] = test_df[col].map(cat_to_id).fillna(0).astype(int)
        if gap_df is not None and len(gap_df) > 0:
            gap_df[col] = gap_df[col].map(cat_to_id).fillna(0).astype(int)

    # C. Fitur Temporal Siklik (Cyclical Time Features)
    feature_cols = ['Amount'] + cat_columns
    if use_cyclical_time:
        for split_df in [train_df, val_df, test_df]:
            hour_float = split_df['datetime'].dt.hour + split_df['datetime'].dt.minute / 60.0
            split_df['hour_sin'] = np.sin(2 * np.pi * hour_float / 24.0).astype(np.float32)
            split_df['hour_cos'] = np.cos(2 * np.pi * hour_float / 24.0).astype(np.float32)
            
            dow = split_df['datetime'].dt.dayofweek
            split_df['dow_sin'] = np.sin(2 * np.pi * dow / 7.0).astype(np.float32)
            split_df['dow_cos'] = np.cos(2 * np.pi * dow / 7.0).astype(np.float32)
            
        if gap_df is not None and len(gap_df) > 0:
            hour_float = gap_df['datetime'].dt.hour + gap_df['datetime'].dt.minute / 60.0
            gap_df['hour_sin'] = np.sin(2 * np.pi * hour_float / 24.0).astype(np.float32)
            gap_df['hour_cos'] = np.cos(2 * np.pi * hour_float / 24.0).astype(np.float32)
            dow = gap_df['datetime'].dt.dayofweek
            gap_df['dow_sin'] = np.sin(2 * np.pi * dow / 7.0).astype(np.float32)
            gap_df['dow_cos'] = np.cos(2 * np.pi * dow / 7.0).astype(np.float32)

        time_cols = ['hour_sin', 'hour_cos', 'dow_sin', 'dow_cos']
        feature_cols = feature_cols + time_cols

    artifacts = {
        "scaler_amount": scaler_amount,
        "categorical_encoders": categorical_encoders,
        "feature_cols": feature_cols,
        "feature_mode": feature_mode
    }
    
    return train_df, val_df, test_df, gap_df, feature_cols, artifacts



In [ ]:
%%writefile graph_builder.py
import torch
import numpy as np
import pandas as pd
from torch_geometric.data import HeteroData
from sklearn.preprocessing import StandardScaler

def graph_level_undersample(df: pd.DataFrame, ratio: float = 1.0, seed: int = 42) -> pd.DataFrame:
    """
    Melakukan Graph-Level Undersampling jika ratio < 1.0.
    Jika ratio == 1.0 (default di exp21c & SHINE), seluruh data dipertahankan
    tanpa merusak topologi asli jaringan.
    """
    if ratio >= 1.0:
        return df
        
    print(f"Menerapkan graph-level undersampling dengan rasio {ratio}...")
    labels = df['Is Fraud?'].values
    fraud_idx = np.flatnonzero(labels == 1)
    normal_idx = np.flatnonzero(labels == 0)
    
    if len(fraud_idx) == 0 or len(normal_idx) == 0:
        return df
        
    desired_normal = int(np.floor(len(normal_idx) * ratio))
    if desired_normal > len(normal_idx):
        desired_normal = len(normal_idx)
        
    rng = np.random.default_rng(seed)
    chosen_normal = rng.choice(normal_idx, size=desired_normal, replace=False)
    
    selected_idx = np.concatenate([chosen_normal, fraud_idx])
    selected_idx.sort()
    
    return df.iloc[selected_idx].reset_index(drop=True)

def build_hetero_graph(df: pd.DataFrame, feature_cols: list, scalers_dict: dict = None) -> tuple:
    """
    Membangun graf heterogen tripartit PyTorch Geometric (HeteroData).
    Nodes:
      - transaction: Fitur transaksi bersih (feature_cols) & label target
      - user: Agregat rata-rata Amount & jumlah transaksi
      - merchant: Agregat rata-rata Amount & jumlah transaksi
    Edges:
      - (user, buys, transaction) & (transaction, rev_buys, user)
      - (merchant, sells, transaction) & (transaction, rev_sells, merchant)

    Anti-Leakage Guard (SPLIT-1 & SPLIT-4):
    Jika scalers_dict is None, scaler untuk User dan Merchant di-fit pada data Train window saat ini.
    Jika scalers_dict disediakan, scaler dari Train digunakan untuk me-transform User dan Merchant
    pada Validation dan Test tanpa membocorkan statistik masa depan.
    """
    data = HeteroData()
    
    # 1. Pemetaan ID User dan Merchant ke integer kontigu
    user_mapping = {uid: i for i, uid in enumerate(df['User'].unique())}
    merchant_mapping = {mid: i for i, mid in enumerate(df['Merchant Name'].unique())}
    
    num_users = len(user_mapping)
    num_merchants = len(merchant_mapping)
    num_transactions = len(df)
    
    # 2. Agregasi fitur User dan Merchant
    user_agg = df.groupby('User').agg(
        user_avg_amount=('Amount', 'mean'),
        user_tx_count=('Amount', 'count')
    ).reset_index()
    
    merchant_agg = df.groupby('Merchant Name').agg(
        merchant_avg_amount=('Amount', 'mean'),
        merchant_tx_count=('Amount', 'count')
    ).reset_index()
    
    # Pastikan urutan sesuai dengan mapping
    user_agg['mapped_id'] = user_agg['User'].map(user_mapping)
    user_agg = user_agg.sort_values('mapped_id').reset_index(drop=True)
    
    merchant_agg['mapped_id'] = merchant_agg['Merchant Name'].map(merchant_mapping)
    merchant_agg = merchant_agg.sort_values('mapped_id').reset_index(drop=True)
    
    # Penskalaan fitur User & Merchant (Strict Anti-Leakage)
    if scalers_dict is None:
        scaler_user = StandardScaler()
        user_feats_scaled = scaler_user.fit_transform(user_agg[['user_avg_amount', 'user_tx_count']])
        
        scaler_merchant = StandardScaler()
        merchant_feats_scaled = scaler_merchant.fit_transform(merchant_agg[['merchant_avg_amount', 'merchant_tx_count']])
        
        out_scalers = {
            'scaler_user': scaler_user,
            'scaler_merchant': scaler_merchant
        }
    else:
        scaler_user = scalers_dict['scaler_user']
        scaler_merchant = scalers_dict['scaler_merchant']
        user_feats_scaled = scaler_user.transform(user_agg[['user_avg_amount', 'user_tx_count']])
        merchant_feats_scaled = scaler_merchant.transform(merchant_agg[['merchant_avg_amount', 'merchant_tx_count']])
        out_scalers = scalers_dict
        
    user_features = torch.tensor(user_feats_scaled, dtype=torch.float)
    merchant_features = torch.tensor(merchant_feats_scaled, dtype=torch.float)
    
    # 3. Fitur dan label Transaction
    tx_features = torch.tensor(df[feature_cols].values, dtype=torch.float)
    tx_labels = torch.tensor(df['Is Fraud?'].values, dtype=torch.float)
    
    # Simpan atribut node
    data['transaction'].x = tx_features
    data['transaction'].y = tx_labels
    data['transaction'].num_nodes = num_transactions
    
    data['user'].x = user_features
    data['user'].num_nodes = num_users
    
    data['merchant'].x = merchant_features
    data['merchant'].num_nodes = num_merchants
    
    # 4. Konstruksi Topologi Edges Bipartit / Tripartit
    user_indices = torch.tensor(df['User'].map(user_mapping).values, dtype=torch.long)
    tx_indices = torch.arange(num_transactions, dtype=torch.long)
    merchant_indices = torch.tensor(df['Merchant Name'].map(merchant_mapping).values, dtype=torch.long)
    
    # Edge User <-> Transaction
    data['user', 'buys', 'transaction'].edge_index = torch.stack([user_indices, tx_indices], dim=0)
    data['transaction', 'rev_buys', 'user'].edge_index = torch.stack([tx_indices, user_indices], dim=0)
    
    # Edge Merchant <-> Transaction
    data['merchant', 'sells', 'transaction'].edge_index = torch.stack([merchant_indices, tx_indices], dim=0)
    data['transaction', 'rev_sells', 'merchant'].edge_index = torch.stack([tx_indices, merchant_indices], dim=0)
    
    return data, out_scalers



In [ ]:
%%writefile model.py
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv, HeteroConv

class HeteroGraphSAGE(nn.Module):
    def __init__(
        self, 
        hidden_channels: int, 
        out_channels: int, 
        num_layers: int, 
        data_metadata: tuple, 
        feature_dims: dict, 
        aggr: str = 'mean',
        dropout: float = 0.2
    ):
        super().__init__()
        
        self.num_layers = num_layers
        self.aggr = aggr
        self.dropout_rate = dropout
        self.dropout = nn.Dropout(dropout) if dropout > 0.0 else nn.Identity()
        
        # Inisialisasi proyeksi linear untuk tiap tipe node
        self.lin_embeddings = nn.ModuleDict()
        self.param_embeddings = nn.ParameterDict()
        for node_type, dim in feature_dims.items():
            if dim == 0:
                self.param_embeddings[node_type] = nn.Parameter(torch.randn(1, hidden_channels))
            else:
                self.lin_embeddings[node_type] = nn.Linear(dim, hidden_channels)
                
        self.convs = nn.ModuleList()
        self.bns = nn.ModuleList()
        
        for _ in range(num_layers):
            conv = HeteroConv({
                edge_type: SAGEConv(
                    in_channels=(-1, -1), # Lazy initialization
                    out_channels=hidden_channels,
                    aggr=aggr
                )
                for edge_type in data_metadata[1]
            })
            self.convs.append(conv)
            
            # BatchNorm untuk node transaction
            self.bns.append(nn.BatchNorm1d(hidden_channels))
            
        # Classifier linear bawaan untuk monitoring val AUPRC selama pelatihan GNN
        self.classifier = nn.Linear(hidden_channels, out_channels)

    def forward(self, x_dict, edge_index_dict, num_nodes_dict, return_embeddings=False):
        # 1. Proyeksi fitur awal
        h_dict = {}
        
        for node_type, x in x_dict.items():
            if node_type in self.lin_embeddings:
                h_dict[node_type] = self.lin_embeddings[node_type](x)
            elif node_type in self.param_embeddings:
                num_nodes = num_nodes_dict[node_type]
                h_dict[node_type] = self.param_embeddings[node_type].expand(num_nodes, -1)
                
        # 2. GraphSAGE Message Passing
        for i in range(self.num_layers):
            h_dict = self.convs[i](h_dict, edge_index_dict)
            
            if 'transaction' in h_dict and h_dict['transaction'].size(0) > 0:
                if h_dict['transaction'].size(0) > 1:
                    h_dict['transaction'] = self.bns[i](h_dict['transaction'])
                h_dict['transaction'] = F.relu(h_dict['transaction'])
                h_dict['transaction'] = self.dropout(h_dict['transaction'])
                
        tx_repr = h_dict['transaction']
        
        if return_embeddings:
            return tx_repr
            
        logits = self.classifier(tx_repr).squeeze(-1)
        probs = torch.sigmoid(logits)
        return probs

def manual_weighted_bce(probs, targets, pos_weight):
    """
    Class-weighted Binary Cross-Entropy loss yang stabil secara numerik.
    """
    eps = 1e-7
    probs = torch.clamp(probs, eps, 1.0 - eps)
    loss = - (pos_weight * targets * torch.log(probs) + (1.0 - targets) * torch.log(1.0 - probs))
    return loss.mean()

def manual_unweighted_bce(probs, targets):
    """
    Unweighted BCE loss untuk monitoring loss objektif alami.
    """
    eps = 1e-7
    probs = torch.clamp(probs, eps, 1.0 - eps)
    loss = - (targets * torch.log(probs) + (1.0 - targets) * torch.log(1.0 - probs))
    return loss.mean()



In [ ]:
%%writefile samplers.py
import torch
from torch_geometric.data import HeteroData
from torch_geometric.loader import NeighborLoader

def calculate_degree_centrality(data: HeteroData, edge_type: tuple):
    """Menghitung degree centrality sederhana untuk edge type tertentu."""
    src, rel, dst = edge_type
    edge_index = data[edge_type].edge_index
    num_nodes = data[src].num_nodes
    degree = torch.zeros(num_nodes, dtype=torch.float)
    degree.scatter_add_(0, edge_index[0], torch.ones(edge_index.shape[1]))
    return degree

def apply_uniform_sampling(data: HeteroData):
    """
    Baseline: Uniform Random Sampling.
    Memberikan probabilitas yang sama (1.0) untuk semua edge.
    """
    first_edge = data.edge_types[0]
    if hasattr(data[first_edge], 'edge_weight') and data[first_edge].edge_weight is not None:
        return data
    for edge_type in data.edge_types:
        num_edges = data[edge_type].edge_index.size(1)
        data[edge_type].edge_weight = torch.ones(num_edges, dtype=torch.float)
    return data

def apply_topology_aware_sampling(data: HeteroData):
    """
    Topology-Aware Sampling (Inverse Log-Degree Centrality - v5 Stabilized).
    Menggunakan inverse sublinear log1p(degree) untuk memprioritaskan edge struktural
    tanpa disparitas bobot yang ekstrem.
    """
    first_edge = data.edge_types[0]
    if hasattr(data[first_edge], 'edge_weight') and data[first_edge].edge_weight is not None:
        return data
    for edge_type in data.edge_types:
        deg_src = calculate_degree_centrality(data, edge_type)
        src_nodes = data[edge_type].edge_index[0]
        
        # Sublinear scaling: log1p(degree)
        log_deg = torch.log1p(deg_src[src_nodes])
        weights = 1.0 / (log_deg + 1e-5)
        min_val = weights.min()
        max_val = weights.max()
        if (max_val - min_val) > 1e-5:
            weights = (weights - min_val) / (max_val - min_val)
        else:
            weights = torch.zeros_like(weights)
        data[edge_type].edge_weight = weights + 0.1
    return data

def apply_importance_based_sampling(data: HeteroData):
    """
    Importance-Based Sampling (Log-Degree Centrality Proxy - v5 Stabilized).
    Menggunakan transformasi sublinear log1p untuk meredam varians sampling
    akibat super-hub merchant.
    """
    first_edge = data.edge_types[0]
    if hasattr(data[first_edge], 'edge_weight') and data[first_edge].edge_weight is not None:
        return data
    for edge_type in data.edge_types:
        deg_src = calculate_degree_centrality(data, edge_type)
        src_nodes = data[edge_type].edge_index[0]
        
        # Sublinear scaling: log1p(degree)
        log_deg = torch.log1p(deg_src[src_nodes])
        min_val = log_deg.min()
        max_val = log_deg.max()
        if (max_val - min_val) > 1e-5:
            weights = (log_deg - min_val) / (max_val - min_val)
        else:
            weights = torch.zeros_like(log_deg)
        data[edge_type].edge_weight = weights + 0.1
    return data

def get_neighbor_loader(
    data: HeteroData, 
    num_neighbors: list, 
    batch_size: int, 
    shuffle: bool = True, 
    sampling_method: str = "uniform",
    input_nodes: torch.Tensor = None
) -> NeighborLoader:
    """
    Membangun PyG NeighborLoader dengan metode sampling yang dipilih.
    """
    if sampling_method == "uniform":
        data = apply_uniform_sampling(data)
    elif sampling_method == "topology":
        data = apply_topology_aware_sampling(data)
    elif sampling_method == "importance":
        data = apply_importance_based_sampling(data)
    else:
        raise ValueError(f"Metode sampling tidak dikenal: {sampling_method}. Pilih 'uniform', 'topology', atau 'importance'.")
        
    if input_nodes is None:
        input_nodes_arg = ('transaction', None)
    else:
        input_nodes_arg = ('transaction', input_nodes)
        
    return NeighborLoader(
        data,
        num_neighbors={edge_type: num_neighbors for edge_type in data.edge_types},
        batch_size=batch_size,
        input_nodes=input_nodes_arg,
        shuffle=shuffle
    )



In [ ]:
%%writefile visualize_results.py
import os
import json
import argparse
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')
from sklearn.metrics import precision_recall_curve, roc_curve

def setup_plot_style():
    plt.rcParams.update({
        'font.size': 11,
        'axes.labelsize': 12,
        'axes.titlesize': 13,
        'xtick.labelsize': 10,
        'ytick.labelsize': 10,
        'legend.fontsize': 10,
        'figure.titlesize': 14,
        'figure.dpi': 300,
        'savefig.dpi': 300,
        'savefig.bbox': 'tight'
    })

def plot_multi_window_metrics(result_dir: Path, output_dir: Path, methods: list):
    """
    Visualisasi Kinerja Multi-Window (SPLIT-2):
    Membandingkan metrik F1, AUCPR, Recall, dan Precision di Window 1, Window 2, Window 3,
    serta menampilkan bar Mean ± Std untuk membuktikan stabilitas temporal model.
    """
    for method in methods:
        csv_path = result_dir / f"exp21c_{method}_multi_window_summary.csv"
        json_path = result_dir / f"exp21c_{method}_results.json"
        if not csv_path.exists() or not json_path.exists():
            continue
            
        df = pd.read_csv(csv_path)
        with open(json_path, 'r') as f:
            full_data = json.load(f)
            
        agg = full_data.get('aggregated_summary_mean_std', {})
        
        # Siapkan label dan metrik
        window_labels = df['window_name'].tolist() + ['Mean ± Std']
        f1_vals = df['test_f1'].tolist() + [agg.get('f1_score', {}).get('mean', 0.0)]
        f1_err = [0.0] * len(df) + [agg.get('f1_score', {}).get('std', 0.0)]
        
        aucpr_vals = df['test_auprc'].tolist() + [agg.get('auprc', {}).get('mean', 0.0)]
        aucpr_err = [0.0] * len(df) + [agg.get('auprc', {}).get('std', 0.0)]
        
        rec_vals = df['test_recall'].tolist() + [agg.get('recall', {}).get('mean', 0.0)]
        rec_err = [0.0] * len(df) + [agg.get('recall', {}).get('std', 0.0)]
        
        prec_vals = df['test_precision'].tolist() + [agg.get('precision', {}).get('mean', 0.0)]
        prec_err = [0.0] * len(df) + [agg.get('precision', {}).get('std', 0.0)]
        
        x = np.arange(len(window_labels))
        width = 0.20
        
        fig, ax = plt.subplots(figsize=(12, 6))
        
        rects1 = ax.bar(x - 1.5*width, f1_vals, width, yerr=f1_err, capsize=5, label='F1-Score', color='#2ca02c', edgecolor='black', linewidth=0.7)
        rects2 = ax.bar(x - 0.5*width, aucpr_vals, width, yerr=aucpr_err, capsize=5, label='AUCPR', color='#1f77b4', edgecolor='black', linewidth=0.7)
        rects3 = ax.bar(x + 0.5*width, rec_vals, width, yerr=rec_err, capsize=5, label='Recall', color='#ff7f0e', edgecolor='black', linewidth=0.7)
        rects4 = ax.bar(x + 1.5*width, prec_vals, width, yerr=prec_err, capsize=5, label='Precision', color='#d62728', edgecolor='black', linewidth=0.7)
        
        ax.set_ylabel('Skor Evaluasi (0.0 - 1.0)')
        ax.set_title(f'EXP21C: Kinerja Walk-Forward Multi-Window ({method.upper()})\n'
                     f'Mean F1: {agg.get("f1_score", {}).get("mean", 0):.4f} ± {agg.get("f1_score", {}).get("std", 0):.4f} | '
                     f'Mean AUCPR: {agg.get("auprc", {}).get("mean", 0):.4f} ± {agg.get("auprc", {}).get("std", 0):.4f}',
                     fontweight='bold')
        ax.set_xticks(x)
        ax.set_xticklabels(window_labels, fontweight='bold')
        ax.legend(loc='upper right', frameon=True)
        ax.grid(axis='y', linestyle='--', alpha=0.5)
        ax.set_ylim(0.0, 1.05)
        
        # Highlight kolom rata-rata
        ax.axvspan(len(window_labels) - 1.5, len(window_labels) - 0.5, color='gray', alpha=0.1, zorder=0)
        
        plt.tight_layout()
        save_path = output_dir / f"exp21c_{method}_multi_window_performance.png"
        plt.savefig(save_path)
        plt.close()
        print(f"Chart tersimpan: {save_path}")

def plot_cross_window_pr_roc(result_dir: Path, output_dir: Path, methods: list):
    """
    Visualisasi Kurva PR dan ROC Antar Window:
    Menampilkan kestabilan kurva diskriminasi model lintas era waktu.
    """
    for method in methods:
        npz_path = result_dir / f"exp21c_{method}_predictions.npz"
        json_path = result_dir / f"exp21c_{method}_results.json"
        if not npz_path.exists() or not json_path.exists():
            continue
            
        data = np.load(npz_path)
        with open(json_path, 'r') as f:
            res = json.load(f)
            
        windows = res.get('window_details', {})
        colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']
        
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
        
        color_idx = 0
        for wid, winfo in windows.items():
            prob_key = f"{wid}_test_probs"
            label_key = f"{wid}_test_labels"
            if prob_key not in data or label_key not in data:
                continue
                
            y_probs = data[prob_key]
            y_true = data[label_key]
            if len(np.unique(y_true)) < 2:
                continue
                
            wname = winfo.get('window_name', wid)
            c = colors[color_idx % len(colors)]
            color_idx += 1
            
            # Kurva PR
            prec, rec, _ = precision_recall_curve(y_true, y_probs)
            aucpr = winfo.get('metrics_threshold_independent', {}).get('auprc', 0.0)
            ax1.plot(rec, prec, lw=2.2, color=c, label=f'{wname} (AUCPR = {aucpr:.4f})')
            
            # Kurva ROC
            fpr, tpr, _ = roc_curve(y_true, y_probs)
            roc_auc = winfo.get('metrics_threshold_independent', {}).get('roc_auc', 0.0)
            ax2.plot(fpr, tpr, lw=2.2, color=c, label=f'{wname} (ROC-AUC = {roc_auc:.4f})')
            
        # Format PR plot
        ax1.set_xlabel('Recall')
        ax1.set_ylabel('Precision')
        ax1.set_title(f'EXP21C: Kurva Precision-Recall ({method.upper()})', fontweight='bold')
        ax1.legend(loc='lower left', frameon=True)
        ax1.grid(True, linestyle='--', alpha=0.6)
        ax1.set_xlim([0.0, 1.0])
        ax1.set_ylim([0.0, 1.05])
        
        # Format ROC plot
        ax2.plot([0, 1], [0, 1], color='navy', lw=1.5, linestyle='--', label='Random Chance')
        ax2.set_xlabel('False Positive Rate')
        ax2.set_ylabel('True Positive Rate')
        ax2.set_title(f'EXP21C: Kurva ROC ({method.upper()})', fontweight='bold')
        ax2.legend(loc='lower right', frameon=True)
        ax2.grid(True, linestyle='--', alpha=0.6)
        ax2.set_xlim([0.0, 1.0])
        ax2.set_ylim([0.0, 1.05])
        
        plt.tight_layout()
        save_path = output_dir / f"exp21c_{method}_cross_window_pr_roc.png"
        plt.savefig(save_path)
        plt.close()
        print(f"Chart tersimpan: {save_path}")

def plot_embargo_gap_sensitivity(result_dir: Path, output_dir: Path, methods: list):
    """
    Visualisasi SPLIT-3: Sensitivitas Embargo Gap (0, 8, 30 hari).
    Menguji pengaruh durasi jeda investigasi fraud terhadap performa model di masa depan.
    """
    for method in methods:
        csv_path = result_dir / f"exp21c_{method}_embargo_ablation.csv"
        if not csv_path.exists():
            continue
            
        df = pd.read_csv(csv_path)
        if len(df) == 0:
            continue
            
        gaps = df['embargo_gap_days'].tolist()
        f1_vals = df['test_f1'].tolist()
        rec_vals = df['test_recall'].tolist()
        prec_vals = df['test_precision'].tolist()
        aucpr_vals = df['test_auprc'].tolist()
        
        fig, ax = plt.subplots(figsize=(9, 5.5))
        
        ax.plot(gaps, f1_vals, marker='o', lw=2.5, color='#2ca02c', label='F1-Score')
        ax.plot(gaps, aucpr_vals, marker='s', lw=2.5, color='#1f77b4', label='AUCPR')
        ax.plot(gaps, rec_vals, marker='^', lw=2.0, color='#ff7f0e', linestyle='--', label='Recall')
        ax.plot(gaps, prec_vals, marker='v', lw=2.0, color='#d62728', linestyle='--', label='Precision')
        
        ax.set_xlabel('Durasi Embargo Gap (Hari Transaksi Dilewati)')
        ax.set_ylabel('Skor Metrik Evaluasi')
        ax.set_title(f'EXP21C: Pengaruh Durasi Embargo Gap terhadap Kinerja ({method.upper()})\n'
                     f'Ablasi SPLIT-3 (0 hari = Instant, 8 hari = SHINE, 30 hari = Siklus Billing)',
                     fontweight='bold')
        ax.set_xticks(gaps)
        ax.set_xticklabels([f'{g} Hari' for g in gaps], fontweight='bold')
        ax.legend(loc='lower left', frameon=True)
        ax.grid(True, linestyle='--', alpha=0.6)
        ax.set_ylim(0.0, 1.05)
        
        plt.tight_layout()
        save_path = output_dir / f"exp21c_{method}_embargo_gap_sensitivity.png"
        plt.savefig(save_path)
        plt.close()
        print(f"Chart tersimpan: {save_path}")

def plot_feature_subsets_lift(result_dir: Path, output_dir: Path, methods: list):
    """
    Visualisasi Nilai Tambah Representasi GraphSAGE (Lift):
    Membandingkan Tabular Only vs GraphSAGE Only vs Hybrid pada seluruh Window.
    """
    for method in methods:
        csv_path = result_dir / f"exp21c_{method}_feature_subsets.csv"
        if not csv_path.exists():
            continue
            
        df = pd.read_csv(csv_path)
        if len(df) == 0:
            continue
            
        windows = df['window_name'].unique().tolist()
        subsets = ["Tabular Only", "GraphSAGE Only", "Hybrid (Tabular + GNN)"]
        colors = ['#ff7f0e', '#1f77b4', '#2ca02c']
        
        x = np.arange(len(windows))
        width = 0.25
        
        fig, ax = plt.subplots(figsize=(11, 6))
        
        for i, (sub, c) in enumerate(zip(subsets, colors)):
            sub_df = df[df['subset_name'] == sub]
            f1_scores = [sub_df[sub_df['window_name'] == w]['test_f1'].values[0] if len(sub_df[sub_df['window_name'] == w]) > 0 else 0.0 for w in windows]
            offset = (i - 1) * width
            rects = ax.bar(x + offset, f1_scores, width, label=sub, color=c, edgecolor='black', linewidth=0.7)
            
            # Tambahkan label angka di atas bar
            for rect in rects:
                h = rect.get_height()
                if h > 0:
                    ax.annotate(f'{h:.3f}',
                                xy=(rect.get_x() + rect.get_width() / 2, h),
                                xytext=(0, 3), textcoords="offset points",
                                ha='center', va='bottom', fontsize=8, fontweight='bold')
                                
        ax.set_ylabel('F1-Score pada Test Set')
        ax.set_title(f'EXP21C: Perbandingan Nilai Tambah Representasi GraphSAGE Lintas Waktu ({method.upper()})\n'
                     f'Tabular Only vs GraphSAGE Only vs Hybrid Model', fontweight='bold')
        ax.set_xticks(x)
        ax.set_xticklabels(windows, fontweight='bold')
        ax.legend(loc='upper right', frameon=True)
        ax.grid(axis='y', linestyle='--', alpha=0.6)
        ax.set_ylim(0.0, 1.05)
        
        plt.tight_layout()
        save_path = output_dir / f"exp21c_{method}_gnn_lift_across_windows.png"
        plt.savefig(save_path)
        plt.close()
        print(f"Chart tersimpan: {save_path}")

def plot_gain_distribution_across_windows(result_dir: Path, output_dir: Path, methods: list):
    """
    Visualisasi Porsi Gain Keputusan (Tabular vs GNN) Lintas Jendela Waktu:
    Membuktikan bahwa representasi graf 128-d konsisten berkontribusi di seluruh era.
    """
    for method in methods:
        csv_path = result_dir / f"exp21c_{method}_multi_window_summary.csv"
        if not csv_path.exists():
            continue
            
        df = pd.read_csv(csv_path)
        if len(df) == 0:
            continue
            
        windows = df['window_name'].tolist()
        tab_gain = df['pct_tabular_gain'].tolist()
        gnn_gain = df['pct_gnn_gain'].tolist()
        
        x = np.arange(len(windows))
        width = 0.45
        
        fig, ax = plt.subplots(figsize=(9, 5.5))
        
        p1 = ax.bar(x, tab_gain, width, label='Tabular Features', color='#ff7f0e', edgecolor='black', linewidth=0.7)
        p2 = ax.bar(x, gnn_gain, width, bottom=tab_gain, label='GraphSAGE Representation', color='#1f77b4', edgecolor='black', linewidth=0.7)
        
        for idx in range(len(windows)):
            ax.text(x[idx], tab_gain[idx] / 2, f"{tab_gain[idx]:.1f}%", ha='center', va='center', color='black', fontweight='bold')
            ax.text(x[idx], tab_gain[idx] + (gnn_gain[idx] / 2), f"{gnn_gain[idx]:.1f}%", ha='center', va='center', color='white', fontweight='bold')
            
        ax.set_ylabel('Persentase Split Gain Keputusan (%)')
        ax.set_title(f'EXP21C: Distribusi Gain Keputusan XGBoost Lintas Window ({method.upper()})', fontweight='bold')
        ax.set_xticks(x)
        ax.set_xticklabels(windows, fontweight='bold')
        ax.legend(loc='lower right', frameon=True)
        ax.set_ylim(0, 105)
        ax.grid(axis='y', linestyle='--', alpha=0.5)
        
        plt.tight_layout()
        save_path = output_dir / f"exp21c_{method}_gain_distribution_across_windows.png"
        plt.savefig(save_path)
        plt.close()
        print(f"Chart tersimpan: {save_path}")

def main():
    parser = argparse.ArgumentParser(description="EXP21C Result Visualizer")
    parser.add_argument("--result-dir", type=str, default="../../result/exp21c", help="Direktori hasil EXP21C")
    parser.add_argument("--output-dir", type=str, default=None, help="Direktori penyimpanan plot PNG")
    parser.add_argument("--methods", type=str, default="uniform,topology,importance", help="Metode sampling yang divisualisasikan")
    args = parser.parse_args()
    
    result_dir = Path(args.result_dir)
    output_dir = Path(args.output_dir) if args.output_dir else (result_dir / "plots")
    output_dir.mkdir(parents=True, exist_ok=True)
    
    setup_plot_style()
    methods = [m.strip() for m in args.methods.split(',')]
    
    print("\n" + "="*75)
    print("  MEMULAI GENERASI VISUALISASI HASIL EXP21C (PUBLIKASI 300 DPI)")
    print("="*75)
    
    plot_multi_window_metrics(result_dir, output_dir, methods)
    plot_cross_window_pr_roc(result_dir, output_dir, methods)
    plot_embargo_gap_sensitivity(result_dir, output_dir, methods)
    plot_feature_subsets_lift(result_dir, output_dir, methods)
    plot_gain_distribution_across_windows(result_dir, output_dir, methods)
    
    print("\n" + "="*75)
    print(f"  SELURUH VISUALISASI SELESAI DIBUAT DI: {output_dir}")
    print("="*75 + "\n")

if __name__ == "__main__":
    main()



In [ ]:
%%writefile main.py
import argparse
import time
import json
import gc
from pathlib import Path
import itertools
import numpy as np
import pandas as pd
import torch
import torch.optim as optim
from sklearn.metrics import (
    f1_score, recall_score, precision_score, 
    accuracy_score, roc_auc_score, average_precision_score, 
    confusion_matrix
)
from xgboost import XGBClassifier

from config import load_config
from data_preprocessing import load_raw_data_era_modern, extract_window_splits, apply_strict_anti_leakage_preprocessing
from graph_builder import build_hetero_graph, graph_level_undersample
from samplers import get_neighbor_loader
from model import HeteroGraphSAGE, manual_weighted_bce, manual_unweighted_bce

def extract_features(model, loader, device, total_samples, orig_feat_dim, emb_dim):
    """
    Mengekstrak embedding GNN dan menggabungkannya dengan fitur tabular transaksi yang bersih.
    Menggunakan pre-alokasi memori numpy untuk efisiensi RAM maksimal.
    """
    model.eval()
    final_features = np.empty((total_samples, orig_feat_dim + emb_dim), dtype=np.float32)
    all_targets = np.empty((total_samples,), dtype=np.int64)
    
    start_time = time.time()
    current_idx = 0
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            num_nodes_dict = {ntype: batch[ntype].num_nodes for ntype in batch.node_types}
            
            embeddings = model(
                batch.x_dict, 
                batch.edge_index_dict, 
                num_nodes_dict=num_nodes_dict, 
                return_embeddings=True
            )
            
            batch_size = batch['transaction'].batch_size
            next_idx = min(current_idx + batch_size, total_samples)
            actual_count = next_idx - current_idx
            
            final_features[current_idx:next_idx, :orig_feat_dim] = batch['transaction'].x[:actual_count].cpu().numpy()
            final_features[current_idx:next_idx, orig_feat_dim:] = embeddings[:actual_count].cpu().numpy()
            all_targets[current_idx:next_idx] = batch['transaction'].y[:actual_count].cpu().numpy()
            
            current_idx = next_idx
            
    inference_time = time.time() - start_time
    
    if current_idx < total_samples:
        final_features = final_features[:current_idx]
        all_targets = all_targets[:current_idx]
        
    return final_features, all_targets, inference_time

def find_optimal_threshold(val_probs, val_labels, thresholds=None, min_precision=0.10):
    """
    Mencari threshold optimal tau* yang memaksimalkan F1-score pada Validation set,
    dengan konstrain presisi minimum yang masuk akal secara operasional perbankan.
    """
    if thresholds is None:
        thresholds = np.arange(0.02, 0.70, 0.005)
        
    best_tau = 0.5
    best_f1 = 0.0
    best_metrics = {}
    
    for tau in thresholds:
        preds = (val_probs >= tau).astype(int)
        f1 = f1_score(val_labels, preds, zero_division=0)
        prec = precision_score(val_labels, preds, zero_division=0)
        rec = recall_score(val_labels, preds, zero_division=0)
        
        if f1 > best_f1 and prec >= min_precision:
            best_f1 = f1
            best_tau = float(tau)
            best_metrics = {
                "tau": float(tau),
                "val_f1": float(f1),
                "val_precision": float(prec),
                "val_recall": float(rec)
            }
            
    if best_f1 == 0.0:
        for tau in thresholds:
            preds = (val_probs >= tau).astype(int)
            f1 = f1_score(val_labels, preds, zero_division=0)
            prec = precision_score(val_labels, preds, zero_division=0)
            rec = recall_score(val_labels, preds, zero_division=0)
            if f1 > best_f1:
                best_f1 = f1
                best_tau = float(tau)
                best_metrics = {
                    "tau": float(tau),
                    "val_f1": float(f1),
                    "val_precision": float(prec),
                    "val_recall": float(rec)
                }

    if best_f1 == 0.0:
        best_tau = 0.5
        best_metrics = {
            "tau": 0.5,
            "val_f1": 0.0,
            "val_precision": 0.0,
            "val_recall": 0.0
        }
        
    return best_tau, best_metrics

def evaluate_gnn(model, loader, device, pos_weight):
    """
    Mengevaluasi representasi GNN pada validation loader.
    """
    model.eval()
    total_loss = 0.0
    total_loss_unw = 0.0
    total_samples = 0
    all_probs = []
    all_targets = []
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            num_nodes_dict = {ntype: batch[ntype].num_nodes for ntype in batch.node_types}
            probs = model(batch.x_dict, batch.edge_index_dict, num_nodes_dict=num_nodes_dict)
            batch_size_current = batch['transaction'].batch_size
            probs = probs[:batch_size_current]
            targets = batch['transaction'].y[:batch_size_current]
            
            loss = manual_weighted_bce(probs, targets, pos_weight)
            loss_unw = manual_unweighted_bce(probs, targets)
            
            total_loss += loss.item() * targets.size(0)
            total_loss_unw += loss_unw.item() * targets.size(0)
            total_samples += targets.size(0)
            
            all_probs.append(probs.cpu().numpy())
            all_targets.append(targets.cpu().numpy())
            
    val_loss = total_loss / total_samples if total_samples > 0 else 0.0
    val_loss_unw = total_loss_unw / total_samples if total_samples > 0 else 0.0
    all_probs = np.concatenate(all_probs)
    all_targets = np.concatenate(all_targets)
    
    auprc = float(average_precision_score(all_targets, all_probs)) if len(np.unique(all_targets)) > 1 else 0.0
    roc_auc = float(roc_auc_score(all_targets, all_probs)) if len(np.unique(all_targets)) > 1 else 0.0
    
    return val_loss, val_loss_unw, auprc, roc_auc

def fit_xgb_model(params, train_features, train_labels, val_features, val_labels, use_gpu=True):
    """
    Helper untuk melatih model XGBoost dengan dukungan GPU/CPU fallback.
    """
    eval_set = [(val_features, val_labels)]
    xgb_params = dict(params)
    xgb_params['eval_metric'] = 'aucpr'
    xgb_params['random_state'] = 42
    
    if use_gpu and torch.cuda.is_available():
        try:
            model = XGBClassifier(**xgb_params, tree_method='hist', device='cuda')
            model.fit(train_features, train_labels, eval_set=eval_set, verbose=False)
            return model
        except Exception:
            pass
            
    model = XGBClassifier(**xgb_params, tree_method='hist', n_jobs=-1)
    model.fit(train_features, train_labels, eval_set=eval_set, verbose=False)
    return model

def calculate_gain_distribution(model, feature_names):
    """
    Menghitung split gain untuk fitur Tabular vs 128-d GraphSAGE.
    """
    booster = model.get_booster()
    score_gain = booster.get_score(importance_type='gain')
    score_weight = booster.get_score(importance_type='weight')
    
    total_gain_all = 0.0
    total_gain_tab = 0.0
    total_gain_gnn = 0.0
    importance_list = []
    
    for idx, name in enumerate(feature_names):
        f_key = f"f{idx}"
        gain = float(score_gain.get(f_key, 0.0))
        weight = int(score_weight.get(f_key, 0))
        is_gnn = name.startswith("gnn_emb_")
        total_gain_all += gain
        if is_gnn:
            total_gain_gnn += gain
        else:
            total_gain_tab += gain
            
        importance_list.append({
            "feature_index": idx,
            "feature_name": name,
            "is_gnn": is_gnn,
            "gain": gain,
            "weight": weight
        })
        
    importance_list.sort(key=lambda x: x['gain'], reverse=True)
    pct_tab = (total_gain_tab / total_gain_all * 100.0) if total_gain_all > 0 else 0.0
    pct_gnn = (total_gain_gnn / total_gain_all * 100.0) if total_gain_all > 0 else 0.0
    
    return {
        "total_gain_all": total_gain_all,
        "total_gain_tabular": total_gain_tab,
        "total_gain_gnn": total_gain_gnn,
        "pct_tabular_gain": pct_tab,
        "pct_gnn_gain": pct_gnn,
        "top_features": importance_list[:25]
    }

def main():
    parser = argparse.ArgumentParser(description="EXP21C: Multi-Window Temporal Splitting & Walk-Forward Validation")
    parser.add_argument("--config", type=str, default="config.reference.yaml", help="Path ke file konfigurasi YAML")
    parser.add_argument("--sampling", type=str, default=None, choices=["uniform", "topology", "importance"], help="Metode Neighbor Sampling")
    parser.add_argument("--windows", type=str, default="all", help="Daftar window ID (koma-terpisah) atau 'all'")
    parser.add_argument("--skip-embargo-ablation", action="store_true", help="Lewati studi ablasi embargo gap")
    parser.add_argument("--nrows", type=int, default=None, help="Batas baris dataset untuk smoke test")
    parser.add_argument("--epochs", type=int, default=None, help="Override jumlah epoch GNN")
    parser.add_argument("--device", type=str, default=None, help="Device komputasi: 'cuda' atau 'cpu'")
    args = parser.parse_args()

    # 1. Load Konfigurasi
    cfg = load_config(args.config)
    
    sampling_method = args.sampling or cfg.get('sampling', {}).get('method', 'uniform')
    device_str = args.device or ('cuda' if torch.cuda.is_available() else 'cpu')
    device = torch.device(device_str)
    
    model_dir = Path(cfg.get('output', {}).get('model_dir', 'model/exp21c'))
    result_dir = Path(cfg.get('output', {}).get('result_dir', 'result/exp21c'))
    model_dir.mkdir(parents=True, exist_ok=True)
    result_dir.mkdir(parents=True, exist_ok=True)

    print("="*85)
    print("  EKSPERIMEN 21C (EXP21C): MULTI-WINDOW TEMPORAL SPLITTING & WALK-FORWARD VALIDATION")
    print("="*85)
    print(f"  • Metode Neighbor Sampling : {sampling_method.upper()}")
    print(f"  • Device                   : {device} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})")
    print(f"  • Konfigurasi File         : {args.config}")
    print(f"  • Direktori Hasil          : {result_dir}")
    print("="*85)

    # 2. Pemuatan Data Mentah Era Modern (Year >= 2016) SATU KALI
    csv_path = cfg['data']['transactions']
    start_year = cfg['data'].get('start_year', 2016)
    nrows = args.nrows or cfg['data'].get('max_rows', None)

    t0_data = time.time()
    raw_df = load_raw_data_era_modern(csv_path, start_year=start_year, nrows=nrows)
    data_load_time = time.time() - t0_data

    # 3. Tentukan Windows yang Akan Dijalankan
    all_windows_cfg = cfg.get('time_windows', {}).get('windows', [])
    if args.windows != "all":
        selected_ids = [w.strip() for w in args.windows.split(',')]
        windows_to_run = [w for w in all_windows_cfg if w['id'] in selected_ids]
    else:
        windows_to_run = all_windows_cfg

    print(f"\n[ORCHESTRATOR] Menjalankan {len(windows_to_run)} Temporal Windows:")
    for w in windows_to_run:
        print(f"  - {w['name']} (ID: {w['id']}, T0: {w['reference_date']}, Train: {w['train_days']}d, Test: {w['test_days']}d)")

    # Struktur Penyimpanan Hasil
    all_window_results = {}
    all_predictions = {}
    all_gain_distributions = {}
    all_feature_subsets = []
    
    # Base XGBoost Parameters (Terbukti Stabil & Optimal dari Temuan EXP21B)
    xgb_base_params = {
        'n_estimators': cfg.get('classifier', {}).get('n_estimators', 100),
        'max_depth': cfg.get('classifier', {}).get('max_depth', 4),
        'learning_rate': cfg.get('classifier', {}).get('learning_rate', 0.05),
        'min_child_weight': cfg.get('classifier', {}).get('min_child_weight', 10),
        'subsample': cfg.get('classifier', {}).get('subsample', 0.8),
        'colsample_bytree': cfg.get('classifier', {}).get('colsample_bytree', 0.8)
    }

    # 4. Loop Eksekusi Tiap Jendela Waktu (Walk-Forward Temporal Validation)
    total_start_time = time.time()
    
    for win_idx, win_cfg in enumerate(windows_to_run):
        win_id = win_cfg['id']
        win_name = win_cfg['name']
        ref_date = win_cfg['reference_date']
        tr_days = win_cfg.get('train_days', 300)
        vl_days = win_cfg.get('val_days', 30)
        gp_days = win_cfg.get('gap_days', 8)
        ts_days = win_cfg.get('test_days', 60)
        
        print("\n" + "#"*85)
        print(f"  >>> MEMULAI EKSEKUSI: {win_name.upper()} (T0: {ref_date}) <<<")
        print("#"*85)

        # A. Partisi Temporal strictly chronologic
        train_df, val_df, test_df, gap_df, split_stats = extract_window_splits(
            raw_df, 
            reference_date=ref_date,
            train_days=tr_days,
            val_days=vl_days,
            gap_days=gp_days,
            test_days=ts_days,
            window_name=win_name
        )

        # B. Strict Anti-Leakage Preprocessing (SPLIT-1: Scaler & OOV Dictionary Fit ONLY on Train)
        feature_mode = cfg.get('features', {}).get('mode', 'clean')
        use_cyclical = cfg.get('features', {}).get('use_cyclical_time', True) if isinstance(cfg.get('features', {}).get('use_cyclical_time'), bool) else True
        
        train_df, val_df, test_df, gap_df, feature_cols, prep_artifacts = apply_strict_anti_leakage_preprocessing(
            train_df, val_df, test_df, gap_df,
            feature_mode=feature_mode,
            use_cyclical_time=use_cyclical
        )

        # C. Bangun Graf Heterogen dengan Entity Scaler Guard
        train_graph, train_scalers = build_hetero_graph(train_df, feature_cols, scalers_dict=None)
        val_graph, _ = build_hetero_graph(val_df, feature_cols, scalers_dict=train_scalers)
        test_graph, _ = build_hetero_graph(test_df, feature_cols, scalers_dict=train_scalers)

        # D. PyG NeighborLoaders
        num_neighbors = cfg['training'].get('num_neighbors', [15, 10])
        batch_size = cfg['training'].get('batch_size', 1024)
        
        train_loader = get_neighbor_loader(train_graph, num_neighbors, batch_size, shuffle=True, sampling_method=sampling_method)
        val_loader = get_neighbor_loader(val_graph, num_neighbors, batch_size, shuffle=False, sampling_method=sampling_method)
        test_loader = get_neighbor_loader(test_graph, num_neighbors, batch_size, shuffle=False, sampling_method=sampling_method)

        # E. Inisialisasi & Latih HeteroGraphSAGE
        hidden_channels = cfg['model'].get('hidden_channels', 128)
        num_layers = cfg['model'].get('num_layers', 2)
        aggr = cfg['model'].get('aggr', 'mean')
        dropout = cfg['model'].get('dropout', 0.2)
        
        feature_dims = {
            'transaction': train_graph['transaction'].x.size(1),
            'user': train_graph['user'].x.size(1),
            'merchant': train_graph['merchant'].x.size(1)
        }
        
        gnn_model = HeteroGraphSAGE(
            hidden_channels=hidden_channels,
            out_channels=1,
            num_layers=num_layers,
            data_metadata=train_graph.metadata(),
            feature_dims=feature_dims,
            aggr=aggr,
            dropout=dropout
        ).to(device)

        # Pembobotan Loss GNN
        n_pos = float(train_graph['transaction'].y.sum().item())
        n_neg = float(len(train_graph['transaction'].y) - n_pos)
        raw_pos_weight = (n_neg / n_pos) if n_pos > 0 else 1.0
        sqrt_pos_weight = float(np.sqrt(raw_pos_weight))
        gnn_loss_pos_weight = min(sqrt_pos_weight, cfg['training'].get('max_pos_weight', 50.0))

        optimizer = optim.Adam(
            gnn_model.parameters(),
            lr=cfg['training'].get('learning_rate', 0.001),
            weight_decay=cfg['training'].get('weight_decay', 0.0001)
        )
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode='max', factor=0.5, patience=cfg['training'].get('lr_scheduler_patience', 3)
        )

        epochs = args.epochs or cfg['training'].get('epochs', 20)
        early_stopping_patience = cfg['training'].get('early_stopping_patience', 7)
        best_val_auprc = 0.0
        patience_counter = 0
        best_gnn_weights = None

        print(f"\n[GNN TRAINING] Melatih HeteroGraphSAGE pada {win_name} ({epochs} epochs)...")
        gnn_start_time = time.time()
        for epoch in range(1, epochs + 1):
            gnn_model.train()
            total_train_loss = 0.0
            total_tx_nodes = 0
            
            for batch in train_loader:
                batch = batch.to(device)
                optimizer.zero_grad()
                num_nodes_dict = {ntype: batch[ntype].num_nodes for ntype in batch.node_types}
                probs = gnn_model(batch.x_dict, batch.edge_index_dict, num_nodes_dict=num_nodes_dict)
                batch_size_cur = batch['transaction'].batch_size
                probs = probs[:batch_size_cur]
                targets = batch['transaction'].y[:batch_size_cur]
                
                loss = manual_weighted_bce(probs, targets, gnn_loss_pos_weight)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(gnn_model.parameters(), cfg['training'].get('grad_clip_norm', 5.0))
                optimizer.step()
                
                total_train_loss += loss.item() * targets.size(0)
                total_tx_nodes += targets.size(0)
                
            val_loss, val_loss_unw, val_auprc, val_roc = evaluate_gnn(gnn_model, val_loader, device, gnn_loss_pos_weight)
            scheduler.step(val_auprc)
            
            if val_auprc > best_val_auprc:
                best_val_auprc = val_auprc
                patience_counter = 0
                best_gnn_weights = {k: v.cpu().clone() for k, v in gnn_model.state_dict().items()}
                marker = " (*)"
            else:
                patience_counter += 1
                marker = ""
                
            if epoch % 2 == 0 or epoch == 1 or marker != "":
                print(f"  • Epoch {epoch:02d}/{epochs:02d} | Train Loss: {total_train_loss/total_tx_nodes:.4f} | Val AUPRC: {val_auprc:.4f} | Val ROC: {val_roc:.4f}{marker}")
                
            if patience_counter >= early_stopping_patience:
                print(f"  [EARLY STOPPING] Pelatihan GNN dihentikan pada epoch {epoch} (Best Val AUPRC: {best_val_auprc:.4f})")
                break
                
        gnn_train_time = time.time() - gnn_start_time
        if best_gnn_weights is not None:
            gnn_model.load_state_dict({k: v.to(device) for k, v in best_gnn_weights.items()})

        # F. Ekstraksi Fitur Gabungan (128-d GNN + Clean Tabular)
        print(f"\n[FEATURE EXTRACTION] Mengekstrak 128-d representasi GraphSAGE untuk {win_name}...")
        orig_feat_dim = len(feature_cols)
        
        train_features, train_labels, _ = extract_features(gnn_model, train_loader, device, len(train_df), orig_feat_dim, hidden_channels)
        val_features, val_labels, _ = extract_features(gnn_model, val_loader, device, len(val_df), orig_feat_dim, hidden_channels)
        test_features, test_labels, test_inf_time = extract_features(gnn_model, test_loader, device, len(test_df), orig_feat_dim, hidden_channels)

        feature_names = feature_cols + [f"gnn_emb_{i}" for i in range(hidden_channels)]

        # G. Pelatihan & Evaluasi XGBoost Ter-Regularisasi
        print(f"\n[XGBOOST CLASSIFIER] Melatih XGBoost regularized (SPW={sqrt_pos_weight:.2f}, Depth=4, MCW=10)...")
        xgb_params = dict(xgb_base_params)
        xgb_params['scale_pos_weight'] = float(round(sqrt_pos_weight, 2))
        
        t0_xgb = time.time()
        xgb_model = fit_xgb_model(xgb_params, train_features, train_labels, val_features, val_labels, use_gpu=True)
        xgb_train_time = time.time() - t0_xgb

        # Dynamic Threshold Calibration pada Validation Set
        val_probs = xgb_model.predict_proba(val_features)[:, 1]
        min_prec_constraint = cfg.get('threshold_calibration', {}).get('min_precision_constraint', 0.10)
        best_tau, val_tau_metrics = find_optimal_threshold(val_probs, val_labels, min_precision=min_prec_constraint)
        
        # Evaluasi pada Out-of-Sample Test Set (60 Hari)
        test_probs = xgb_model.predict_proba(test_features)[:, 1]
        
        # Metrik Bebas Threshold
        test_auprc = float(average_precision_score(test_labels, test_probs)) if len(np.unique(test_labels)) > 1 else 0.0
        test_roc_auc = float(roc_auc_score(test_labels, test_probs)) if len(np.unique(test_labels)) > 1 else 0.0
        
        # Metrik Operasional Perbankan
        top_1pct_count = max(1, int(round(len(test_labels) * 0.01)))
        top_1pct_idx = np.argsort(test_probs)[::-1][:top_1pct_count]
        precision_1pct = float(test_labels[top_1pct_idx].mean())
        base_fraud_rate = float(test_labels.mean())
        lift_1pct = float(precision_1pct / base_fraud_rate) if base_fraud_rate > 0 else 0.0

        top_500_idx = np.argsort(test_probs)[::-1][:500]
        tp_at_500 = int(test_labels[top_500_idx].sum())
        precision_at_500 = float(tp_at_500 / 500)

        # Metrik pada Threshold Optimal tau*
        preds_opt = (test_probs >= best_tau).astype(int)
        f1_opt = float(f1_score(test_labels, preds_opt, zero_division=0))
        rec_opt = float(recall_score(test_labels, preds_opt, zero_division=0))
        prec_opt = float(precision_score(test_labels, preds_opt, zero_division=0))
        acc_opt = float(accuracy_score(test_labels, preds_opt))
        cm_opt = confusion_matrix(test_labels, preds_opt)
        tn_opt, fp_opt, fn_opt, tp_opt = [int(v) for v in cm_opt.ravel()]
        spec_opt = tn_opt / (tn_opt + fp_opt) if (tn_opt + fp_opt) > 0 else 0.0
        gmean_opt = float(np.sqrt(rec_opt * spec_opt))

        # Metrik pada Default Threshold 0.50
        preds_05 = (test_probs >= 0.50).astype(int)
        f1_05 = float(f1_score(test_labels, preds_05, zero_division=0))
        rec_05 = float(recall_score(test_labels, preds_05, zero_division=0))
        prec_05 = float(precision_score(test_labels, preds_05, zero_division=0))

        # Audit Feature Importance Gain
        gain_info = calculate_gain_distribution(xgb_model, feature_names)

        print("\n" + "-"*85)
        print(f"  HASIL EVALUASI {win_name.upper()} PADA TEST SET MASA DEPAN (N={len(test_labels):,})")
        print("-"*85)
        print(f"  • Metrik Bebas Ambang Batas : AUCPR = {test_auprc:.4f} | ROC-AUC = {test_roc_auc:.4f}")
        print(f"  • Metrik Operasional Bank   : Lift@1% = {lift_1pct:.2f}x | TP@500 = {tp_at_500} / 500 ({precision_at_500*100:.1f}%)")
        print(f"  • Threshold Optimal tau*={best_tau:.3f} : F1 = {f1_opt:.4f} | Recall = {rec_opt*100:.2f}% (TP={tp_opt}, FN={fn_opt}) | Prec = {prec_opt*100:.2f}% (FP={fp_opt})")
        print(f"  • Threshold Default 0.50    : F1 = {f1_05:.4f} | Recall = {rec_05*100:.2f}% | Prec = {prec_05*100:.2f}%")
        print(f"  • Feature Split Gain        : Tabular = {gain_info['pct_tabular_gain']:.2f}% | GraphSAGE = {gain_info['pct_gnn_gain']:.2f}%")
        print("-"*85)

        # H. Ablasi Subset Fitur (Tabular vs GNN vs Hybrid) pada Window Ini
        if cfg.get('ablation', {}).get('feature_subsets', True):
            print(f"\n[FEATURE ABLATION] Menguji kontribusi subset fitur pada {win_name}...")
            subsets = {
                "Tabular Only": (train_features[:, :orig_feat_dim], val_features[:, :orig_feat_dim], test_features[:, :orig_feat_dim]),
                "GraphSAGE Only": (train_features[:, orig_feat_dim:], val_features[:, orig_feat_dim:], test_features[:, orig_feat_dim:]),
                "Hybrid (Tabular + GNN)": (train_features, val_features, test_features)
            }
            
            for s_name, (tr_f, vl_f, ts_f) in subsets.items():
                m_sub = fit_xgb_model(xgb_params, tr_f, train_labels, vl_f, val_labels, use_gpu=True)
                vl_p = m_sub.predict_proba(vl_f)[:, 1]
                t_sub, _ = find_optimal_threshold(vl_p, val_labels, min_precision=min_prec_constraint)
                ts_p = m_sub.predict_proba(ts_f)[:, 1]
                
                s_auprc = float(average_precision_score(test_labels, ts_p)) if len(np.unique(test_labels)) > 1 else 0.0
                s_roc = float(roc_auc_score(test_labels, ts_p)) if len(np.unique(test_labels)) > 1 else 0.0
                p_sub = (ts_p >= t_sub).astype(int)
                s_f1 = float(f1_score(test_labels, p_sub, zero_division=0))
                s_rec = float(recall_score(test_labels, p_sub, zero_division=0))
                s_prec = float(precision_score(test_labels, p_sub, zero_division=0))
                
                all_feature_subsets.append({
                    "window_id": win_id,
                    "window_name": win_name,
                    "subset_name": s_name,
                    "optimal_tau": t_sub,
                    "test_f1": s_f1,
                    "test_recall": s_rec,
                    "test_precision": s_prec,
                    "test_auprc": s_auprc,
                    "test_roc_auc": s_roc
                })
                print(f"  • {s_name:24s} | F1: {s_f1:.4f} | Recall: {s_rec*100:.1f}% | Prec: {s_prec*100:.1f}% | AUCPR: {s_auprc:.4f}")

        # Simpan metrik per-window
        window_record = {
            "window_id": win_id,
            "window_name": win_name,
            "reference_date": ref_date,
            "split_statistics": split_stats,
            "optimal_tau": best_tau,
            "metrics_optimal_threshold": {
                "threshold": best_tau,
                "f1_score": f1_opt,
                "recall": rec_opt,
                "precision": prec_opt,
                "accuracy": acc_opt,
                "gmean": gmean_opt,
                "confusion_matrix": {"tn": tn_opt, "fp": fp_opt, "fn": fn_opt, "tp": tp_opt}
            },
            "metrics_default_threshold_05": {
                "threshold": 0.50,
                "f1_score": f1_05,
                "recall": rec_05,
                "precision": prec_05
            },
            "metrics_threshold_independent": {
                "auprc": test_auprc,
                "roc_auc": test_roc_auc,
                "lift_1pct": lift_1pct,
                "tp_at_500": tp_at_500,
                "precision_at_500": precision_at_500
            },
            "feature_importance": {
                "total_gain_tabular": gain_info['total_gain_tabular'],
                "total_gain_gnn": gain_info['total_gain_gnn'],
                "pct_tabular_gain": gain_info['pct_tabular_gain'],
                "pct_gnn_gain": gain_info['pct_gnn_gain']
            },
            "runtime_seconds": {
                "gnn_training": gnn_train_time,
                "xgb_training": xgb_train_time,
                "test_inference": test_inf_time
            }
        }
        all_window_results[win_id] = window_record
        all_gain_distributions[win_id] = gain_info
        
        # Simpan prediksi untuk plotting
        all_predictions[f"{win_id}_test_probs"] = test_probs.astype(np.float32)
        all_predictions[f"{win_id}_test_labels"] = test_labels.astype(np.int8)
        all_predictions[f"{win_id}_val_probs"] = val_probs.astype(np.float32)
        all_predictions[f"{win_id}_val_labels"] = val_labels.astype(np.int8)

        # Bersihkan memori GPU antar jendela
        del train_graph, val_graph, test_graph, train_loader, val_loader, test_loader
        del train_features, val_features, test_features
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    # 5. SPLIT-3: Ablasi Durasi Embargo Gap (0d vs 8d vs 30d) pada Window 2
    embargo_ablation_results = []
    embargo_cfg = cfg.get('time_windows', {}).get('embargo_ablation', {})
    if embargo_cfg.get('enabled', True) and not args.skip_embargo_ablation:
        target_win_id = embargo_cfg.get('target_window_id', 'window_2')
        gap_durations = embargo_cfg.get('gap_durations', [0, 8, 30])
        target_win_cfg = next((w for w in all_windows_cfg if w['id'] == target_win_id), None)
        
        if target_win_cfg is not None:
            print("\n" + "="*85)
            print(f"  >>> MENJALANKAN SPLIT-3: ABLASI DURASI EMBARGO GAP PADA {target_win_cfg['name'].upper()} <<<")
            print("="*85)
            
            ref_date = target_win_cfg['reference_date']
            tr_days = target_win_cfg.get('train_days', 300)
            vl_days = target_win_cfg.get('val_days', 30)
            ts_days = target_win_cfg.get('test_days', 60)
            
            for test_gap in gap_durations:
                print(f"\n--- Pengujian Embargo Gap: {test_gap} hari ---")
                tr_df, vl_df, ts_df, gp_df, sp_st = extract_window_splits(
                    raw_df, reference_date=ref_date,
                    train_days=tr_days, val_days=vl_days,
                    gap_days=test_gap, test_days=ts_days,
                    window_name=f"Gap-{test_gap}d"
                )
                
                tr_df, vl_df, ts_df, gp_df, f_cols, _ = apply_strict_anti_leakage_preprocessing(
                    tr_df, vl_df, ts_df, gp_df, feature_mode=feature_mode, use_cyclical_time=use_cyclical
                )
                
                # Model baseline tabular + waktu siklik (langsung teruji cepat dan akurat)
                xgb_gap = fit_xgb_model(xgb_base_params, tr_df[f_cols].values, tr_df['Is Fraud?'].values, vl_df[f_cols].values, vl_df['Is Fraud?'].values)
                vl_p = xgb_gap.predict_proba(vl_df[f_cols].values)[:, 1]
                t_gap_opt, _ = find_optimal_threshold(vl_p, vl_df['Is Fraud?'].values, min_precision=min_prec_constraint)
                
                ts_p = xgb_gap.predict_proba(ts_df[f_cols].values)[:, 1]
                ts_y = ts_df['Is Fraud?'].values
                
                g_auprc = float(average_precision_score(ts_y, ts_p)) if len(np.unique(ts_y)) > 1 else 0.0
                g_roc = float(roc_auc_score(ts_y, ts_p)) if len(np.unique(ts_y)) > 1 else 0.0
                p_gap = (ts_p >= t_gap_opt).astype(int)
                g_f1 = float(f1_score(ts_y, p_gap, zero_division=0))
                g_rec = float(recall_score(ts_y, p_gap, zero_division=0))
                g_prec = float(precision_score(ts_y, p_gap, zero_division=0))
                
                embargo_ablation_results.append({
                    "target_window_id": target_win_id,
                    "target_window_name": target_win_cfg['name'],
                    "embargo_gap_days": test_gap,
                    "test_start_date": sp_st['splits']['test']['start_date'],
                    "test_end_date": sp_st['splits']['test']['end_date'],
                    "test_transactions": len(ts_df),
                    "test_fraud_count": int(ts_y.sum()),
                    "optimal_tau": t_gap_opt,
                    "test_f1": g_f1,
                    "test_recall": g_rec,
                    "test_precision": g_prec,
                    "test_auprc": g_auprc,
                    "test_roc_auc": g_roc
                })
                print(f"  • Gap {test_gap:2d} hari | Test Fraud={int(ts_y.sum())} | tau*={t_gap_opt:.3f} | F1: {g_f1:.4f} | Recall: {g_rec*100:.2f}% | Prec: {g_prec*100:.2f}% | AUCPR: {g_auprc:.4f}")

    # 6. Agregasi Statistik Lintas Window (Mean ± Std - SPLIT-2)
    f1_list = [w['metrics_optimal_threshold']['f1_score'] for w in all_window_results.values()]
    rec_list = [w['metrics_optimal_threshold']['recall'] for w in all_window_results.values()]
    prec_list = [w['metrics_optimal_threshold']['precision'] for w in all_window_results.values()]
    auprc_list = [w['metrics_threshold_independent']['auprc'] for w in all_window_results.values()]
    roc_list = [w['metrics_threshold_independent']['roc_auc'] for w in all_window_results.values()]
    lift_list = [w['metrics_threshold_independent']['lift_1pct'] for w in all_window_results.values()]
    tp500_list = [w['metrics_threshold_independent']['tp_at_500'] for w in all_window_results.values()]
    gnn_gain_list = [w['feature_importance']['pct_gnn_gain'] for w in all_window_results.values()]

    aggregated_summary = {
        "num_windows_evaluated": len(all_window_results),
        "f1_score": {"mean": float(np.mean(f1_list)), "std": float(np.std(f1_list))},
        "recall": {"mean": float(np.mean(rec_list)), "std": float(np.std(rec_list))},
        "precision": {"mean": float(np.mean(prec_list)), "std": float(np.std(prec_list))},
        "auprc": {"mean": float(np.mean(auprc_list)), "std": float(np.std(auprc_list))},
        "roc_auc": {"mean": float(np.mean(roc_list)), "std": float(np.std(roc_list))},
        "lift_1pct": {"mean": float(np.mean(lift_list)), "std": float(np.std(lift_list))},
        "tp_at_500": {"mean": float(np.mean(tp500_list)), "std": float(np.std(tp500_list))},
        "pct_gnn_gain": {"mean": float(np.mean(gnn_gain_list)), "std": float(np.std(gnn_gain_list))}
    }

    print("\n" + "="*85)
    print("  RANGKUMAN AGREGASI KINERJA MULTI-WINDOW (WALK-FORWARD VALIDATION - EXP21C)")
    print("="*85)
    print(f"  • Metrik F1-Score (Optimal tau*) : {aggregated_summary['f1_score']['mean']:.4f} ± {aggregated_summary['f1_score']['std']:.4f}")
    print(f"  • Metrik Recall                  : {aggregated_summary['recall']['mean']*100:.2f}% ± {aggregated_summary['recall']['std']*100:.2f}%")
    print(f"  • Metrik Precision               : {aggregated_summary['precision']['mean']*100:.2f}% ± {aggregated_summary['precision']['std']*100:.2f}%")
    print(f"  • Metrik AUCPR                   : {aggregated_summary['auprc']['mean']:.4f} ± {aggregated_summary['auprc']['std']:.4f}")
    print(f"  • Metrik ROC-AUC                 : {aggregated_summary['roc_auc']['mean']:.4f} ± {aggregated_summary['roc_auc']['std']:.4f}")
    print(f"  • Lift @ 1% Transaksi Teratas    : {aggregated_summary['lift_1pct']['mean']:.2f}x ± {aggregated_summary['lift_1pct']['std']:.2f}x")
    print(f"  • TP @ 500 Alert Harian          : {aggregated_summary['tp_at_500']['mean']:.1f} ± {aggregated_summary['tp_at_500']['std']:.1f}")
    print(f"  • Kontribusi Gain GraphSAGE      : {aggregated_summary['pct_gnn_gain']['mean']:.2f}% ± {aggregated_summary['pct_gnn_gain']['std']:.2f}%")
    print("="*85)

    # 7. Simpan Seluruh Berkas Hasil Eksekusi
    total_pipeline_time = time.time() - total_start_time

    final_results_output = {
        "experiment_id": "EXP21C",
        "experiment_name": "Multi-Window Temporal Splitting & Walk-Forward Validation",
        "sampling_method": sampling_method,
        "base_xgb_parameters": xgb_base_params,
        "aggregated_summary_mean_std": aggregated_summary,
        "window_details": all_window_results,
        "embargo_gap_ablation": embargo_ablation_results,
        "runtime_total_seconds": total_pipeline_time
    }

    # A. JSON Utama
    results_json_path = result_dir / f"exp21c_{sampling_method}_results.json"
    with open(results_json_path, 'w', encoding='utf-8') as f:
        json.dump(final_results_output, f, indent=4)
    print(f"\n[OUTPUT] File hasil utama tersimpan di: {results_json_path}")

    # B. CSV Ringkasan Multi-Window
    mw_rows = []
    for wid, wres in all_window_results.items():
        mw_rows.append({
            "window_id": wid,
            "window_name": wres['window_name'],
            "reference_date": wres['reference_date'],
            "optimal_tau": wres['optimal_tau'],
            "test_f1": wres['metrics_optimal_threshold']['f1_score'],
            "test_recall": wres['metrics_optimal_threshold']['recall'],
            "test_precision": wres['metrics_optimal_threshold']['precision'],
            "test_auprc": wres['metrics_threshold_independent']['auprc'],
            "test_roc_auc": wres['metrics_threshold_independent']['roc_auc'],
            "lift_1pct": wres['metrics_threshold_independent']['lift_1pct'],
            "tp_at_500": wres['metrics_threshold_independent']['tp_at_500'],
            "pct_gnn_gain": wres['feature_importance']['pct_gnn_gain'],
            "pct_tabular_gain": wres['feature_importance']['pct_tabular_gain']
        })
    mw_df = pd.DataFrame(mw_rows)
    mw_csv_path = result_dir / f"exp21c_{sampling_method}_multi_window_summary.csv"
    mw_df.to_csv(mw_csv_path, index=False)
    print(f"[OUTPUT] Tabel CSV multi-window tersimpan di: {mw_csv_path}")

    # C. CSV Feature Subsets Ablation
    if all_feature_subsets:
        fs_df = pd.DataFrame(all_feature_subsets)
        fs_csv_path = result_dir / f"exp21c_{sampling_method}_feature_subsets.csv"
        fs_df.to_csv(fs_csv_path, index=False)
        print(f"[OUTPUT] Tabel CSV subset fitur tersimpan di: {fs_csv_path}")

    # D. CSV Embargo Gap Ablation
    if embargo_ablation_results:
        eg_df = pd.DataFrame(embargo_ablation_results)
        eg_csv_path = result_dir / f"exp21c_{sampling_method}_embargo_ablation.csv"
        eg_df.to_csv(eg_csv_path, index=False)
        print(f"[OUTPUT] Tabel CSV embargo gap tersimpan di: {eg_csv_path}")

    # E. Prediksi NPZ Terkompresi
    pred_npz_path = result_dir / f"exp21c_{sampling_method}_predictions.npz"
    np.savez_compressed(pred_npz_path, **all_predictions)
    print(f"[OUTPUT] Array prediksi terkompresi tersimpan di: {pred_npz_path}")

    print("\n" + "="*85)
    print("  🎉 PIPELINE EKSPERIMEN 21C SELESAI DENGAN SUKSES! 🎉")
    print("="*85)

if __name__ == "__main__":
    main()



In [ ]:
%%writefile run_all.py
import os
import subprocess
import sys
import json
import yaml
from pathlib import Path

def run_script(script_name, args=[]):
    print(f"\n{'='*75}")
    print(f"  >>> MENJALANKAN SCRIPT: {script_name} {' '.join(args)}")
    print(f"{'='*75}\n")
    
    command = [sys.executable, script_name] + args
    result = subprocess.run(command)
    
    if result.returncode != 0:
        print(f"\n[ERROR] Eksekusi {script_name} gagal dengan kode {result.returncode}!")
        sys.exit(result.returncode)
    else:
        print(f"\n[SUCCESS] {script_name} selesai dieksekusi.\n")

if __name__ == "__main__":
    print("="*75)
    print("  MEMULAI RUN ALL: EKSPERIMEN 21C (MULTI-WINDOW TEMPORAL SPLITTING)")
    print("="*75)
    
    # 1. Install dependencies jika di environment Kaggle/Colab
    if os.path.exists("/kaggle/input") or os.path.exists("/content"):
        print("\n[INFO] Menginstall dependencies (torch_geometric & pyg-lib)...")
        try:
            import torch
            pt_version = torch.__version__.split('+')[0]
            cuda_version = torch.version.cuda.replace('.', '') if torch.cuda.is_available() and torch.version.cuda else 'cpu'
            if cuda_version != 'cpu':
                url = f"https://data.pyg.org/whl/torch-{pt_version}+cu{cuda_version}.html"
                os.system(f"pip install torch_geometric pyg_lib torch_scatter torch_sparse -f {url} > /dev/null 2>&1")
            else:
                os.system("pip install torch_geometric > /dev/null 2>&1")
        except Exception as e:
            print(f"[WARNING] Fallback install torch_geometric: {e}")
            os.system("pip install torch_geometric > /dev/null 2>&1")

    # 2. Deteksi path konfigurasi
    if os.path.exists("/kaggle/input"):
        config_path = "config.kaggle.yaml"
        csv_path = None
        for root, dirs, files in os.walk('/kaggle/input'):
            if 'credit_card_transactions-ibm_v2.csv' in files:
                csv_path = os.path.join(root, 'credit_card_transactions-ibm_v2.csv')
                break
        if csv_path:
            print(f"[INFO] Dataset ditemukan di: {csv_path}")
            with open(config_path, 'r') as f:
                cfg = yaml.safe_load(f)
            cfg['data']['transactions'] = csv_path
            with open(config_path, 'w') as f:
                yaml.dump(cfg, f)
    elif os.path.exists("/content"):
        config_path = "config.colab.yaml"
    else:
        config_path = "config.reference.yaml"

    with open(config_path, 'r') as f:
        cfg_loaded = yaml.safe_load(f)
    res_dir = Path(cfg_loaded.get('output', {}).get('result_dir', 'result/exp21c'))
    res_dir.mkdir(parents=True, exist_ok=True)

    # 3. Jalankan eksperimen untuk 3 metode sampling
    sampling_methods = ["uniform", "topology", "importance"]
    pass_through_args = sys.argv[1:]
    
    for method in sampling_methods:
        print(f"\n{'*'*75}")
        print(f"  >>> MULAI EXP21C DENGAN SAMPLING: {method.upper()}")
        print(f"{'*'*75}\n")
        cmd_args = ["--config", config_path, "--sampling", method] + pass_through_args
        run_script("main.py", cmd_args)
        
    # 4. Generate Visualisasi Hasil (Chart Ilmiah 300 DPI)
    if os.path.exists("visualize_results.py"):
        print("\n" + "="*75)
        print("  >>> MENJALANKAN GENERASI VISUALISASI HASIL EXP21C")
        print("="*75 + "\n")
        try:
            plots_dir = res_dir / 'plots'
            run_script("visualize_results.py", ["--result-dir", str(res_dir), "--output-dir", str(plots_dir)])
        except Exception as e:
            print(f"[WARNING] Gagal menjalankan visualisasi otomatis: {e}")

    # 5. Penyusunan Ringkasan Komparatif Master Markdown (Bab 4 Skripsi)
    summary_report_path = res_dir / "exp21c_summary_report.md"
    summary_lines = [
        "# Ringkasan Hasil Eksperimen 21C (EXP21C): Multi-Window Temporal Splitting & Walk-Forward Validation\n\n",
        "Dokumen ini memuat rangkuman hasil evaluasi komparatif model pada multi-horizon waktu modern (2017 s/d 2020) ",
        "serta ablasi durasi embargo gap dan nilai tambah representasi GraphSAGE.\n\n",
        "## 1. Rangkuman Kinerja Walk-Forward Multi-Window (Mean ± Std)\n\n",
        "| Sampler Method | Test F1-Score (μ ± σ) | Test Recall (μ ± σ) | Test Precision (μ ± σ) | Test AUCPR (μ ± σ) | Test ROC-AUC (μ ± σ) | Lift @ 1% (μ ± σ) | TP @ 500 (μ ± σ) | GNN Gain Share (%) |\n",
        "| :--- | :---: | :---: | :---: | :---: | :---: | :---: | :---: | :---: |\n"
    ]

    for method in sampling_methods:
        res_file = res_dir / f"exp21c_{method}_results.json"
        if res_file.exists():
            with open(res_file, 'r', encoding='utf-8') as f:
                d = json.load(f)
            agg = d.get('aggregated_summary_mean_std', {})
            f1_m, f1_s = agg.get('f1_score', {}).get('mean', 0.0), agg.get('f1_score', {}).get('std', 0.0)
            rec_m, rec_s = agg.get('recall', {}).get('mean', 0.0) * 100, agg.get('recall', {}).get('std', 0.0) * 100
            prec_m, prec_s = agg.get('precision', {}).get('mean', 0.0) * 100, agg.get('precision', {}).get('std', 0.0) * 100
            auc_m, auc_s = agg.get('auprc', {}).get('mean', 0.0), agg.get('auprc', {}).get('std', 0.0)
            roc_m, roc_s = agg.get('roc_auc', {}).get('mean', 0.0), agg.get('roc_auc', {}).get('std', 0.0)
            lft_m, lft_s = agg.get('lift_1pct', {}).get('mean', 0.0), agg.get('lift_1pct', {}).get('std', 0.0)
            tp_m, tp_s = agg.get('tp_at_500', {}).get('mean', 0.0), agg.get('tp_at_500', {}).get('std', 0.0)
            gnn_m, gnn_s = agg.get('pct_gnn_gain', {}).get('mean', 0.0), agg.get('pct_gnn_gain', {}).get('std', 0.0)
            
            summary_lines.append(
                f"| **{method.capitalize()}** | **{f1_m:.4f} ± {f1_s:.4f}** | {rec_m:.2f}% ± {rec_s:.2f}% | {prec_m:.2f}% ± {prec_s:.2f}% | **{auc_m:.4f} ± {auc_s:.4f}** | {roc_m:.4f} ± {roc_s:.4f} | {lft_m:.2f}x ± {lft_s:.2f}x | {tp_m:.1f} ± {tp_s:.1f} | {gnn_m:.1f}% ± {gnn_s:.1f}% |\n"
            )

    summary_lines.append("\n## 2. Rincian Kinerja per Jendela Waktu (Window 1, 2, 3)\n\n")
    summary_lines.append("| Sampler | Window Name | Periode Horizon | Test Transactions | Fraud Count | Optimal tau* | Test F1-Score | Test Recall | Test Precision | Test AUCPR | GNN Gain Share |\n")
    summary_lines.append("| :--- | :--- | :--- | :---: | :---: | :---: | :---: | :---: | :---: | :---: | :---: |\n")

    for method in sampling_methods:
        csv_file = res_dir / f"exp21c_{method}_multi_window_summary.csv"
        if csv_file.exists():
            import pandas as pd
            df_w = pd.read_csv(csv_file)
            for _, row in df_w.iterrows():
                summary_lines.append(
                    f"| {method.capitalize()} | **{row['window_name']}** | {row['reference_date']} | - | - | {row['optimal_tau']:.3f} | **{row['test_f1']:.4f}** | {row['test_recall']*100:.2f}% | {row['test_precision']*100:.2f}% | **{row['test_auprc']:.4f}** | {row['pct_gnn_gain']:.2f}% |\n"
                )

    summary_lines.append("\n## 3. Hasil Ablasi Durasi Embargo Gap (SPLIT-3)\n\n")
    summary_lines.append("| Embargo Gap | Test Period | Test Fraud | Optimal tau* | Test F1-Score | Test Recall | Test Precision | Test AUCPR |\n")
    summary_lines.append("| :---: | :--- | :---: | :---: | :---: | :---: | :---: | :---: |\n")

    default_eg_csv = res_dir / "exp21c_uniform_embargo_ablation.csv"
    if default_eg_csv.exists():
        import pandas as pd
        df_eg = pd.read_csv(default_eg_csv)
        for _, row in df_eg.iterrows():
            summary_lines.append(
                f"| **{row['embargo_gap_days']} Hari** | {row['test_start_date'][:10]} s/d {row['test_end_date'][:10]} | {row['test_fraud_count']} | {row['optimal_tau']:.3f} | **{row['test_f1']:.4f}** | {row['test_recall']*100:.2f}% | {row['test_precision']*100:.2f}% | **{row['test_auprc']:.4f}** |\n"
            )

    summary_lines.append("\n\n*Catatan: Evaluasi menggunakan protokol strictly out-of-sample tanpa kebocoran atribut, penskalaan hanya fit pada train window masing-masing.*")
    
    with open(summary_report_path, 'w', encoding='utf-8') as f:
        f.writelines(summary_lines)
    print(f"\n[INFO] Laporan ringkasan komparasi tersimpan di: {summary_report_path}")

    print("\n" + "="*75)
    print("  🎉 SELURUH PROSES MULTI-WINDOW EXP21C SELESAI DENGAN SUKSES! 🎉")
    print("="*75)



In [ ]:
# Jalankan eksekusi end-to-end 3 metode sampling
!python run_all.py